# EDA — Sepsis-3 Cohort & Discharge Notes

Exploratory analysis of MIMIC-IV discharge summaries linked to the Sepsis-3 cohort.
Examines note structure, demographics, and data quality.

**No patient data is written to disk or displayed in outputs.**

In [113]:
import os
import gzip
import csv
import pandas as pd
import re
from tqdm import tqdm

In [62]:

MIMIC_HOSP = "../data/raw/mimic-iv-3.1/hosp"
MIMIC_ICU  = "../data/raw/mimic-iv-3.1/icu"
MIMIC_NOTEEVENTS = "../data/raw/mimic-iv-3.1/note" 

TABLES = {
    # hosp
    "patients":         MIMIC_HOSP,
    "admissions":       MIMIC_HOSP,
    "labevents":        MIMIC_HOSP,
    "d_labitems":       MIMIC_HOSP,
    "pharmacy":         MIMIC_HOSP,
    "diagnoses_icd":    MIMIC_HOSP,
    "d_icd_diagnoses":  MIMIC_HOSP,
    "procedures_icd":   MIMIC_HOSP,
    "d_icd_procedures": MIMIC_HOSP,
    "omr":              MIMIC_HOSP,
    "discharge":         MIMIC_HOSP,
    # icu
    "icustays":         MIMIC_ICU,
    "chartevents":      MIMIC_ICU,
    "d_items":          MIMIC_ICU,
    "inputevents":      MIMIC_ICU,
    "outputevents":     MIMIC_ICU,
    # notes
    "discharge":         MIMIC_NOTEEVENTS,
}
 
def get_columns(path: str, name: str) -> list[str]:
    filepath = os.path.join(path, f"{name}.csv.gz")
    with gzip.open(filepath, "rt", newline="") as f:
        reader = csv.reader(f)
        return next(reader)  # just the header row
 
print(f"{'Table':<25} Columns")
print("-" * 80)
for name, path in TABLES.items():
    try:
        cols = get_columns(path, name)
        print(f"{name:<25} {cols}")
    except FileNotFoundError:
        print(f"{name:<25} *** FILE NOT FOUND ***")
    except Exception as e:
        print(f"{name:<25} *** ERROR: {e} ***")

Table                     Columns
--------------------------------------------------------------------------------
patients                  ['subject_id', 'gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod']
admissions                ['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'admit_provider_id', 'admission_location', 'discharge_location', 'insurance', 'language', 'marital_status', 'race', 'edregtime', 'edouttime', 'hospital_expire_flag']
labevents                 ['labevent_id', 'subject_id', 'hadm_id', 'specimen_id', 'itemid', 'order_provider_id', 'charttime', 'storetime', 'value', 'valuenum', 'valueuom', 'ref_range_lower', 'ref_range_upper', 'flag', 'priority', 'comments']
d_labitems                ['itemid', 'label', 'fluid', 'category']
pharmacy                  ['subject_id', 'hadm_id', 'pharmacy_id', 'poe_id', 'starttime', 'stoptime', 'medication', 'proc_type', 'status', 'entertime', 'verifiedtime', 'route', 'frequency', 'di

In [29]:
filepath = os.path.join(MIMIC_NOTEEVENTS, "discharge.csv.gz")
df = pd.read_csv(filepath, compression='gzip')
df.columns.to_list()

['note_id',
 'subject_id',
 'hadm_id',
 'note_type',
 'note_seq',
 'charttime',
 'storetime',
 'text']

In [63]:
print("Number of unique patients:", df["subject_id"].nunique())
print("Number of unique hospital admissions:", df["hadm_id"].nunique())
print("Shape of the dataframe:", df.shape)

Number of unique patients: 145914
Number of unique hospital admissions: 331793
Shape of the dataframe: (331793, 8)


In [64]:
cohort = pd.read_csv("../data/cohort/sepsis_cohort.csv", usecols=["subject_id", "hadm_id","sepsis_onset_time"])

In [65]:
print("Number of unique patients:", cohort["subject_id"].nunique())
print("Number of unique hospital admissions:", cohort["hadm_id"].nunique())
print("Shape of the dataframe:", cohort.shape)

Number of unique patients: 25570
Number of unique hospital admissions: 31174
Shape of the dataframe: (32899, 3)


In [67]:
data = cohort.merge(df, on = ["subject_id", "hadm_id"], how = "inner")
print("Shape of the merged dataframe:", data.shape)

Shape of the merged dataframe: (32513, 9)


In [68]:
data["note_length"] = data["text"].str.len()
data["note_length"].describe()

count    32513.000000
mean     13987.466613
std       6103.089041
min        781.000000
25%       9721.000000
50%      12846.000000
75%      16905.000000
max      58596.000000
Name: note_length, dtype: float64

In [69]:

admissions = pd.read_csv(os.path.join(MIMIC_HOSP, "admissions.csv.gz"), compression="gzip", usecols=["subject_id", "hadm_id", "insurance", "race"])
merged = data.merge(
    admissions,
    on=["subject_id", "hadm_id"],
    how="left"
)

merged["insurance"].value_counts()


insurance
Medicare     19279
Private       7540
Medicaid      4469
Other          791
No charge        1
Name: count, dtype: int64

In [70]:
merged["race"].value_counts()

race
WHITE                                        20998
UNKNOWN                                       3032
BLACK/AFRICAN AMERICAN                        2851
OTHER                                         1027
WHITE - OTHER EUROPEAN                         556
UNABLE TO OBTAIN                               434
HISPANIC/LATINO - PUERTO RICAN                 413
ASIAN                                          373
WHITE - RUSSIAN                                343
ASIAN - CHINESE                                331
HISPANIC OR LATINO                             310
HISPANIC/LATINO - DOMINICAN                    214
BLACK/CAPE VERDEAN                             203
PATIENT DECLINED TO ANSWER                     182
BLACK/CARIBBEAN ISLAND                         171
PORTUGUESE                                     164
ASIAN - SOUTH EAST ASIAN                       134
BLACK/AFRICAN                                  113
ASIAN - ASIAN INDIAN                            85
WHITE - EASTERN EUROPEAN  

## Note structure
1. Starts with deidentified info like name, DOB,etc (___ usage for deidentification). Service and allergies
2. Chief complaint, major surgical or invasive procedure
3. History of present illness + medical, social and family history
4. Physical Examination, admission labs, studies and tests
5. Brief hospital course
6. Medications on admission
7. Discharge info: medications in bullet points, disposition, diagnosis, condition details, instructions


In [96]:
section_headers = [
    "chief complaint",
    "major surgical or invasive procedure",
    "history of present illness",
    "past medical history",
    "social history",
    "family history",
    "physical exam",
    "pertinent results",
    "brief hospital course",
    "medications on admission",
    "discharge medications",
    "discharge disposition",
    "discharge diagnosis",
    "discharge condition",
    "discharge instructions"
]

In [100]:
def extract_sections(text: str) -> dict[str, str]:
    text = text.lower()
    sections = {}
    for i, header in enumerate(section_headers):
        pattern = rf"{re.escape(header)}:(.*?)(?={'|'.join(re.escape(h) + ':' for h in section_headers)}|$)"
        match = re.search(pattern, text, re.DOTALL)
        if match:
            sections[header] = match.group(1).strip()
    return sections


In [115]:
tqdm.pandas(desc = "Extracting sections")
merged["sections"] = merged["text"].progress_apply(extract_sections)


Extracting sections: 100%|██████████| 32513/32513 [00:15<00:00, 2097.45it/s]


In [ ]:
sections_df = pd.json_normalize(merged["sections"])
merged = pd.concat([merged.drop(columns="sections"), sections_df], axis=1)


In [125]:
merged.to_parquet("../data/processed/merged_sepsis_notes.parquet", index=False)